# Train a new Mask R-CNN with NEURAL

Train a new model on a folder of labeled ROIs (`<stem>_summary_rgb.npy` + `<stem>_instance_masks.npy` pairs in the same folder). Most knobs are configurable via a JSON-style dict below.

Output: a new run dir under `NEURAL/nn/models/<run_name>/` containing `checkpoints/best_model.pth`, `config.json`, `history.csv`, `training_curves.png`.

## 1. Training config

**Most-edited fields:**
1. `data.training_dir` – folder of `summary_rgb` + `instance_masks` pairs
2. `run_name` – becomes `NEURAL/nn/models/<run_name>/`
3. `training.epochs`, `learning_rate`, `batch_size`
4. `model.training_pixel_size_um` – the µm/px your training images were acquired at

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

config = {
    'run_name': 'my_new_iglusnfr_model',

    'data': {
        'training_dir':  'C:/path/to/training/folder',   # <-- EDIT ME
        'val_fraction':  0.0,                            # 0 = no held-out validation
    },

    'training': {
        'epochs':              30,
        'batch_size':          4,
        'learning_rate':       1e-3,
        'early_stop_patience': 0,            # 0 = disabled; otherwise stop after N epochs without val improvement
        'seed':                42,
        'num_workers':         0,
        'augment':             True
    },

    'model': {
        'approach':                'flowfield_production',
        'pretrained_encoder':      True,
        'apply_imagenet_norm':     True,
        'training_pixel_size_um':  0.223,    # µm/px your training images were acquired at
        'patch_size':              384,
        'patches_per_image':       8,
        'auto_pos_weight_fg':      True,
        'auto_pos_weight_cap':     5.0,
        'pos_weight_fg':           5.0
    },

    'loss': {
        'bce_weight':       0.5,
        'tversky_weight':   0.5,
        'tversky_alpha':    0.7,
        'tversky_beta':     0.3,
        'flow_weight':      2.0
    },

    'flowfield': {
        'fg_threshold':           0.7,
        'n_steps':                10,
        'step_size':              3.0,
        'min_distance':           12,
        'vote_threshold':         8.0,
        'min_roi_px':             32,
        'watershed_compactness':  0.0
    },

    'validation': {
        'tile_size':         384,
        'tile_overlap':      192,
        'save_val_overlays': False,           # native-res overlays are matplotlib-OOM hazardous
        'max_val_overlays':  4
    }
}
print(json.dumps(config, indent=2))

## 2. Verify training data

Each summary image needs a matching mask file: `<stem>_summary_rgb.npy` + `<stem>_instance_masks.npy`.

In [ ]:
td = Path(config['data']['training_dir'])
summaries = sorted(td.glob('*_summary_rgb.npy'))
matched = [s for s in summaries if (s.parent / s.name.replace('_summary_rgb.npy', '_instance_masks.npy')).exists()]
print(f'training_dir   : {td}')
print(f'  summaries    : {len(summaries)}')
print(f'  with masks   : {len(matched)}')
if len(summaries) and not matched:
    print('  WARN: no matching _instance_masks.npy files found')

## 3. Train

Output run dir: `NEURAL/nn/models/<run_name>/`. After training, this dir can be referenced via `nn.model = '<run_name>'` in `analyze_dataset.ipynb` (drop the `_production` suffix in that path).

In [ ]:
from NEURAL.nn.scripts.approach_flowfield import train_run
from NEURAL.nn.scripts.config import Config

flat = dict(
    training_dir=config['data']['training_dir'],
    val_fraction=config['data']['val_fraction'],
    run_name=config['run_name'],
    epochs=config['training']['epochs'],
    batch_size=config['training']['batch_size'],
    lr=config['training']['learning_rate'],
    early_stop_patience=config['training']['early_stop_patience'],
    seed=config['training']['seed'],
    num_workers=config['training']['num_workers'],
    augment=config['training']['augment'],
    approach=config['model']['approach'],
    pretrained_encoder=config['model']['pretrained_encoder'],
    apply_imagenet_norm=config['model']['apply_imagenet_norm'],
    training_pixel_size_um=config['model']['training_pixel_size_um'],
    patch_size=config['model']['patch_size'],
    patches_per_image=config['model']['patches_per_image'],
    auto_pos_weight_fg=config['model']['auto_pos_weight_fg'],
    auto_pos_weight_cap=config['model']['auto_pos_weight_cap'],
    pos_weight_fg=config['model']['pos_weight_fg'],
    loss_fg_bce_weight=config['loss']['bce_weight'],
    loss_fg_tversky_weight=config['loss']['tversky_weight'],
    tversky_alpha=config['loss']['tversky_alpha'],
    tversky_beta=config['loss']['tversky_beta'],
    loss_flow_weight=config['loss']['flow_weight'],
    ff_fg_threshold=config['flowfield']['fg_threshold'],
    ff_n_steps=config['flowfield']['n_steps'],
    ff_step_size=config['flowfield']['step_size'],
    ff_min_distance=config['flowfield']['min_distance'],
    ff_vote_threshold=config['flowfield']['vote_threshold'],
    ff_min_roi_px=config['flowfield']['min_roi_px'],
    watershed_compactness=config['flowfield']['watershed_compactness'],
    val_tile_size=config['validation']['tile_size'],
    val_tile_overlap=config['validation']['tile_overlap'],
    save_val_overlays=config['validation']['save_val_overlays'],
    max_val_overlays=config['validation']['max_val_overlays'],
    output_root=str(Path.cwd() / 'nn' / 'models'),
)
cfg = Config(**flat)
run_dir = train_run(cfg)
print('\\nrun_dir:', run_dir)

## 4. After training

Your new checkpoint lives at `NEURAL/nn/models/<run_name>/checkpoints/best_model.pth`. To use it in `analyze_dataset.ipynb`, point `nn.model` at it via a sibling `<run_name>_production/` dir, OR adjust the path resolver in `NEURAL/figure_engine/render.py::_model_dir`.